## 한국어 뉴스 주제 분류 파인튜닝

기본 모델: `klue/bert-base`  
데이터셋: `klue/ynat`  
결과: 한국어 뉴스 제목을 주제별로 분류

파인튜닝 전에는 기본 모델이 뉴스 주제 분류용으로 학습되어 있지 않기 때문에 원하는 라벨을 제대로 예측하지 못한다.  
파인튜닝 후에는 한국어 뉴스 제목을 입력했을 때 정치, 경제, 사회, 생활문화, 세계, IT과학, 스포츠 중 하나로 분류할 수 있다.

영화리뷰 감정분석
기본 모델: beomi/kcbert-base
데이터셋: NSMC
분류 라벨: 부정, 긍정
결과: 한국어 영화 리뷰의 감정을 분류

In [1]:
!pip install -q -U "datasets<4.0.0" transformers evaluate accelerate

In [28]:
import torch
import numpy as np
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    pipeline
)
import evaluate

print("GPU 사용 가능 여부:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU 이름:", torch.cuda.get_device_name(0))

GPU 사용 가능 여부: True
GPU 이름: Tesla T4


데이터 불러오기

In [29]:
from datasets import load_dataset

dataset = load_dataset("valurank/News_Articles_Categorization", trust_remote_code=True)

dataset
# put code here

DatasetDict({
    train: Dataset({
        features: ['Text', 'Category'],
        num_rows: 3722
    })
})

기본 모델과 토크나이저 불러오기

In [30]:
# put code here
dataset["train"][0]


{'Text': 'Elon Musk, Amber Heard Something\'s Fishy On Wrapped-Up Sushi Last we heard, Elon Musk and Amber Heard were "not back together" even though they kiss goodbye and dance real close ... sorry, we\'re not buying that now. Amber and Elon went on a sushi date Monday in WeHo, and looked like the full-blown hand-holding couple that\'s definitely on again. But that\'s only because that\'s exactly what they are -- no matter how many times they try to say they\'re not reunited. We broke the story ... Elon and Amber started hanging out again this past fall ... after announcing their split in the summer. Since then, they\'ve smooched and gone dancing together. If it looks like a reunited duck, walks like a reunited duck ... TMZ.com',
 'Category': 'Entertainment'}

In [31]:
for i in range(5):
    print("기사:", dataset["train"][i]["Text"])
    print("종류:", dataset["train"][i]["Category"])
    print()

기사: Elon Musk, Amber Heard Something's Fishy On Wrapped-Up Sushi Last we heard, Elon Musk and Amber Heard were "not back together" even though they kiss goodbye and dance real close ... sorry, we're not buying that now. Amber and Elon went on a sushi date Monday in WeHo, and looked like the full-blown hand-holding couple that's definitely on again. But that's only because that's exactly what they are -- no matter how many times they try to say they're not reunited. We broke the story ... Elon and Amber started hanging out again this past fall ... after announcing their split in the summer. Since then, they've smooched and gone dancing together. If it looks like a reunited duck, walks like a reunited duck ... TMZ.com
종류: Entertainment

기사: Scientists are developing more than 100 coronavirus vaccines using a range of techniques, some of which are well-established and some of which have never been approved for medical use before. Most of these vaccines target the so-called spike proteins 

In [32]:
unique_categories = dataset["train"].unique("Category")
num_labels = len(unique_categories)

id2label = {i: label for i, label in enumerate(unique_categories)}
label2id = {label: i for i, label in enumerate(unique_categories)}

def preprocess_labels(examples):
    examples["labels"] = [label2id[category] for category in examples["Category"]]
    return examples

dataset_with_labels = dataset["train"].map(preprocess_labels, batched=True)

splitted_dataset = dataset_with_labels.train_test_split(test_size=0.2, seed=42)
train_dataset = splitted_dataset["train"]
eval_dataset = splitted_dataset["test"]

print(f"훈련 데이터셋 크기: {len(train_dataset)}")
print(f"평가 데이터셋 크기: {len(eval_dataset)}")
print(f"카테고리 개수: {num_labels}")

훈련 데이터셋 크기: 2977
평가 데이터셋 크기: 745
카테고리 개수: 8


In [33]:
model_name = "klue/bert-base"

tokenizer = AutoTokenizer.from_pretrained(model_name)

In [34]:
model_before = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=num_labels, # dynamically set num_labels
    id2label=id2label,
    label2id=label2id
)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: klue/bert-base
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [35]:
before_classifier = pipeline(
    "text-classification",
    model=model_before,
    tokenizer=tokenizer,
    device=0 if torch.cuda.is_available() else -1
)

test_texts = [
    '''Elon Musk, Amber Heard Something's Fishy On Wrapped-Up Sushi Last we heard, Elon Musk and Amber Heard were "not back together" even though they kiss goodbye and dance real close ... sorry, we're not buying that now. Amber and Elon went on a sushi date Monday in WeHo, and looked like the full-blown hand-holding couple that's definitely on again. But that's only because that's exactly what they are -- no matter how many times they try to say they're not reunited. We broke the story ... Elon and Amber started hanging out again this past fall ... after announcing their split in the summer. Since then, they've smooched and gone dancing together. If it looks like a reunited duck, walks like a reunited duck ... TMZ.com''',

    '''Scientists are developing more than 100 coronavirus vaccines using a range of techniques, some of which are well-established and some of which have never been approved for medical use before...''', # (너무 길어서 본문 생략, 본인 코드에서는 그대로 두시면 됩니다!)

    '''Jared Fogle Shut Down By Judge In Bid for Early Release 1/22/2018 Jared Fogle got no love from the judge who sentenced him to 15 years in prison ... ruling his legal arguments were BS...''',

    '''The agency had come under fire from members of Congress and other groups for allowing dozens of wildly inaccurate tests to proliferate without oversight.Credit...David J. Phillip/Associated PressPublished May 4, 2020Updated May 7, 2020The Food and Drug Administration announced on Monday...'''
]

for text in test_texts:
    result = before_classifier(text)
    print("입력:", text)
    print("파인튜닝 전 결과:", result)
    print()

입력: Elon Musk, Amber Heard Something's Fishy On Wrapped-Up Sushi Last we heard, Elon Musk and Amber Heard were "not back together" even though they kiss goodbye and dance real close ... sorry, we're not buying that now. Amber and Elon went on a sushi date Monday in WeHo, and looked like the full-blown hand-holding couple that's definitely on again. But that's only because that's exactly what they are -- no matter how many times they try to say they're not reunited. We broke the story ... Elon and Amber started hanging out again this past fall ... after announcing their split in the summer. Since then, they've smooched and gone dancing together. If it looks like a reunited duck, walks like a reunited duck ... TMZ.com
파인튜닝 전 결과: [{'label': 'science', 'score': 0.16130954027175903}]

입력: Scientists are developing more than 100 coronavirus vaccines using a range of techniques, some of which are well-established and some of which have never been approved for medical use before...
파인튜닝 전 결과: 

In [36]:
def tokenize_function(examples):
    tokenized_inputs = tokenizer(
        examples["Text"],
        padding="max_length",
        truncation=True,
        max_length=64
    )
    return tokenized_inputs

tokenized_train = train_dataset.map(tokenize_function, batched=True)
tokenized_eval = eval_dataset.map(tokenize_function, batched=True)

Map:   0%|          | 0/745 [00:00<?, ? examples/s]

In [37]:
print(tokenized_train.column_names)

['Text', 'Category', 'labels', 'input_ids', 'token_type_ids', 'attention_mask']


In [38]:
remove_cols = [col for col in ["Text", "Category"] if col in tokenized_train.column_names]
tokenized_train = tokenized_train.remove_columns(remove_cols)
tokenized_eval = tokenized_eval.remove_columns(remove_cols)

# Set format to "torch" here for both train and eval datasets
tokenized_train.set_format("torch")
tokenized_eval.set_format("torch")

print("Tokenized train dataset features:", tokenized_train.column_names)
print("Tokenized eval dataset features:", tokenized_eval.column_names)
# print("First tokenized train example:", tokenized_train[0]) # Removed to fix ImportError

Tokenized train dataset features: ['labels', 'input_ids', 'token_type_ids', 'attention_mask']
Tokenized eval dataset features: ['labels', 'input_ids', 'token_type_ids', 'attention_mask']


In [39]:
model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=num_labels, # dynamically set num_labels
    id2label=id2label,
    label2id=label2id
)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: klue/bert-base
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [40]:
accuracy = evaluate.load("accuracy")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return accuracy.compute(predictions=predictions, references=labels)

In [52]:
training_args = TrainingArguments(
    output_dir="./nsmc_klue_bert_model",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=5,
    weight_decay=0.01,
    logging_steps=20,
    report_to="none",
    load_best_model_at_end=True
)

In [53]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_eval,  # Now tokenized_eval is defined
    # tokenizer=tokenizer, # This argument is not valid for Trainer, removed as per previous fix
    compute_metrics=compute_metrics
)

In [54]:
trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy
1,0.421440,0.370398,0.881879
2,0.240992,0.354542,0.895302
3,0.124432,0.401908,0.889933
4,0.092703,0.405740,0.904698
5,0.093324,0.438622,0.904698


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=935, training_loss=0.1992074622189935, metrics={'train_runtime': 270.8953, 'train_samples_per_second': 54.947, 'train_steps_per_second': 3.452, 'total_flos': 489577380264960.0, 'train_loss': 0.1992074622189935, 'epoch': 5.0})

In [55]:
trainer.save_model("./my_korean_news_model")
tokenizer.save_pretrained("./my_korean_news_model")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('./my_korean_news_model/tokenizer_config.json',
 './my_korean_news_model/tokenizer.json')

In [56]:
after_classifier = pipeline(
    "text-classification",
    model="./my_korean_news_model",
    tokenizer="./my_korean_news_model",
    device=0 if torch.cuda.is_available() else -1
)

for text in test_texts:
    result = after_classifier(text)
    print("입력:", text)
    print("파인튜닝 후 결과:", result)
    print()

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

입력: Elon Musk, Amber Heard Something's Fishy On Wrapped-Up Sushi Last we heard, Elon Musk and Amber Heard were "not back together" even though they kiss goodbye and dance real close ... sorry, we're not buying that now. Amber and Elon went on a sushi date Monday in WeHo, and looked like the full-blown hand-holding couple that's definitely on again. But that's only because that's exactly what they are -- no matter how many times they try to say they're not reunited. We broke the story ... Elon and Amber started hanging out again this past fall ... after announcing their split in the summer. Since then, they've smooched and gone dancing together. If it looks like a reunited duck, walks like a reunited duck ... TMZ.com
파인튜닝 후 결과: [{'label': 'Entertainment', 'score': 0.9456427693367004}]

입력: Scientists are developing more than 100 coronavirus vaccines using a range of techniques, some of which are well-established and some of which have never been approved for medical use before...
파인튜닝 후

In [57]:
label_map = {
    "Entertainment": "엔터테이먼트",
    "science": "과학",
    "Health": "건강",
    "Politics": "정치",
    "Sports": "스포츠",
    "World": "세계",
    "Tech": "기술",
    "Business": "사업"
}

def compare_prediction(text):
    before = before_classifier(text)[0]
    after = after_classifier(text)[0]

    print("=" * 80)
    print("입력 기사")
    print(text)
    print("-" * 80)

    print("▶ 파인튜닝 전")
    print(f"예측 : {label_map[before['label']]}")
    print(f"확신도 : {before['score']:.2%}")

    print()

    print("▶ 파인튜닝 후")
    print(f"예측 : {label_map[after['label']]}")
    print(f"확신도 : {after['score']:.2%}")

    print("=" * 80)
    print()

compare_prediction(
    "Apple announced its latest artificial intelligence features for the iPhone during its annual developer conference."
)

compare_prediction(
    "The government announced a new policy to reduce housing prices across the country."
)

compare_prediction(
    "Scientists have developed a new vaccine that shows promising results in clinical trials."
)

compare_prediction(
    "Manchester City won the championship after defeating Arsenal in the final match."
)

입력 기사
Apple announced its latest artificial intelligence features for the iPhone during its annual developer conference.
--------------------------------------------------------------------------------
▶ 파인튜닝 전
예측 : 스포츠
확신도 : 16.72%

▶ 파인튜닝 후
예측 : 기술
확신도 : 98.42%

입력 기사
The government announced a new policy to reduce housing prices across the country.
--------------------------------------------------------------------------------
▶ 파인튜닝 전
예측 : 세계
확신도 : 16.70%

▶ 파인튜닝 후
예측 : 기술
확신도 : 32.55%

입력 기사
Scientists have developed a new vaccine that shows promising results in clinical trials.
--------------------------------------------------------------------------------
▶ 파인튜닝 전
예측 : 건강
확신도 : 18.39%

▶ 파인튜닝 후
예측 : 건강
확신도 : 89.81%

입력 기사
Manchester City won the championship after defeating Arsenal in the final match.
--------------------------------------------------------------------------------
▶ 파인튜닝 전
예측 : 엔터테이먼트
확신도 : 16.96%

▶ 파인튜닝 후
예측 : 스포츠
확신도 : 95.76%



In [58]:
my_texts = [
    "Elon Musk announced new AI technology that will be integrated into Tesla vehicles next year.",
    "Scientists are developing a new vaccine that has shown promising results in clinical trials.",
    "The president introduced a new economic policy aimed at reducing inflation.",
    "The national football team secured a dramatic victory in the World Cup qualifying match.",
    "Samsung Electronics unveiled its next-generation semiconductor technology for AI applications."
]

for text in my_texts:
    compare_prediction(text)

입력 기사
Elon Musk announced new AI technology that will be integrated into Tesla vehicles next year.
--------------------------------------------------------------------------------
▶ 파인튜닝 전
예측 : 건강
확신도 : 16.00%

▶ 파인튜닝 후
예측 : 기술
확신도 : 95.63%

입력 기사
Scientists are developing a new vaccine that has shown promising results in clinical trials.
--------------------------------------------------------------------------------
▶ 파인튜닝 전
예측 : 건강
확신도 : 17.81%

▶ 파인튜닝 후
예측 : 건강
확신도 : 92.83%

입력 기사
The president introduced a new economic policy aimed at reducing inflation.
--------------------------------------------------------------------------------
▶ 파인튜닝 전
예측 : 세계
확신도 : 16.29%

▶ 파인튜닝 후
예측 : 정치
확신도 : 85.89%

입력 기사
The national football team secured a dramatic victory in the World Cup qualifying match.
--------------------------------------------------------------------------------
▶ 파인튜닝 전
예측 : 엔터테이먼트
확신도 : 18.86%

▶ 파인튜닝 후
예측 : 스포츠
확신도 : 94.86%

입력 기사
Samsung Electronics unveiled its next-gene